In [1]:
#########################################
# Imports
#########################################

from langchain_community.llms import Ollama
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.vectorstores import FAISS

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory

from langchain_core.output_parsers import StrOutputParser
from langchain.tools import tool

c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#########################################
# LLM
#########################################

llm = Ollama(model="mistral")


C:\Users\User\AppData\Local\Temp\ipykernel_2840\3345290249.py:5: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  llm = Ollama(model="mistral")


In [8]:
#########################################
# Knowledge Base
#########################################

documents = [
"Japan is best visited during spring (March-May) and autumn (September-November).",
"Italy requires a Schengen visa for Indian travelers.",
"Thailand has vegetarian food options including veg pad thai."
]

In [9]:
#########################################
# Chunking
#########################################

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=20
)

chunks = text_splitter.create_documents(documents)



In [11]:

#########################################
# Embeddings
#########################################

embeddings = OllamaEmbeddings(model="mistral")


In [12]:
#########################################
# Vector Store
#########################################

vectorstore = FAISS.from_documents(chunks, embeddings)

retriever = vectorstore.as_retriever()

In [13]:
#########################################
# Retriever Runnable
#########################################

def retrieve_docs(query):

    docs = retriever.invoke(query)

    context = "\n".join([doc.page_content for doc in docs])

    return context


retrieval_runnable = RunnableLambda(retrieve_docs)

In [14]:



#########################################
# TOOLS
#########################################

@tool
def weather_tool(city: str):
    """Return weather information for a city."""

    mock_weather = {
        "Paris": "18°C Cloudy",
        "Tokyo": "22°C Sunny",
        "Rome": "25°C Clear"
    }

    return mock_weather.get(city, "Weather data not available")


@tool
def travel_cost_tool(destination: str):
    """Return estimated travel cost from India."""

    mock_costs = {
        "Italy": "₹80k - ₹1.2L",
        "Japan": "₹90k - ₹1.5L",
        "Thailand": "₹30k - ₹60k"
    }

    return mock_costs.get(destination, "Cost data not available")


In [15]:
#########################################
# Prompt Builder
#########################################

def build_prompt(inputs):

    context = inputs["context"]
    query = inputs["query"]

    prompt = f"""
You are a helpful travel assistant.

Use the context to answer the question.

Context:
{context}

Question:
{query}
"""

    return prompt


prompt_runnable = RunnableLambda(build_prompt)

In [16]:
#########################################
# RAG Pipeline
#########################################

rag_chain = (
    {
        "context": retrieval_runnable,
        "query": RunnablePassthrough()
    }
    | prompt_runnable
    | llm
    | StrOutputParser()
)

In [17]:
#########################################
# SESSION MEMORY
#########################################

store = {}

def get_session_history(session_id):

    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()

    return store[session_id]


In [18]:
#########################################
# Wrap chain with memory
#########################################

chat_chain = RunnableWithMessageHistory(
    rag_chain,
    get_session_history,
    input_messages_key="query",
    history_messages_key="chat_history"
)

In [21]:
#########################################
# Entity Tracking
#########################################

entity_store = {}

def update_entities(session_id, query):

    if session_id not in entity_store:
        entity_store[session_id] = {
            "diet": None,
            "destination": None
        }

    q = query.lower()

    if "vegetarian" in q:
        entity_store[session_id]["diet"] = "vegetarian"

    if "vegan" in q:
        entity_store[session_id]["diet"] = "vegan"

    if "italy" in q:
        entity_store[session_id]["destination"] = "Italy"

    if "japan" in q:
        entity_store[session_id]["destination"] = "Japan"

    if "thailand" in q:
        entity_store[session_id]["destination"] = "Thailand"


In [20]:
#########################################
# Ask Assistant
#########################################

def ask_assistant(session_id, query):

    update_entities(session_id, query)

    q = query.lower()

    # Tool routing

    if "weather" in q:

        if "paris" in q:
            return weather_tool.invoke({"city": "Paris"})

        if "tokyo" in q:
            return weather_tool.invoke({"city": "Tokyo"})

        if "rome" in q:
            return weather_tool.invoke({"city": "Rome"})


    if "cost" in q or "price" in q:

        if "italy" in q:
            return travel_cost_tool.invoke({"destination": "Italy"})

        if "japan" in q:
            return travel_cost_tool.invoke({"destination": "Japan"})

        if "thailand" in q:
            return travel_cost_tool.invoke({"destination": "Thailand"})


    response = chat_chain.invoke(
        {"query": query},
        config={"configurable": {"session_id": session_id}}
    )

    return response



In [7]:
print(ask_assistant("user1", "Best time to visit Japan"))

 The best times to visit Japan are during spring (March-May) and autumn (September-November). These seasons offer beautiful cherry blossoms and vibrant fall foliage, making for a picturesque travel experience.
